# Azure AI Foundry를 활용한 RAG 구현

## 1. 환경 설정 및 라이브러리 임포트

### 인증 (Identity & Credentials)
- DefaultAzureCredential: 환경 변수, VS Code 로그인 정보, Azure CLI 로그인 정보 등을 순차적으로 확인하여 자동으로 인증을 처리
- AzureKeyCredential: 전통적인 'API Key' 방식

### 검색 (Azure AI Search)

먼저 ai foundry에서 ai search 리소스를 생성해야 합니다.

- SearchClient: 데이터를 검색(Query)하거나, 문서를 업로드(Indexing)하는 작업을 수행.
- SearchIndexClient: 인덱스를 생성, 수정, 삭제하는 작업을 수행. 데이터가 저장될 **구조(Schema)**를 정의.
    - SearchIndex: 인덱스 전체 정의
    - SimpleField: 필터링이나 정렬에 쓰이는 단순 데이터 (예: ID, 날짜, 숫자)
    - SearchableField: 형태소 분석을 통해 검색이 가능한 텍스트 데이터 (예: 본문 내용)


In [ ]:
import os
import time
from pypdf import PdfReader
from langchain_community.document_loaders import PyMuPDFLoader

from dotenv import load_dotenv

# Azure SDK
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchFieldDataType, SearchableField,
)
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage

load_dotenv("../../env")

endpoint       = os.getenv("END_POINT", "").rstrip("/")
api_key        = os.getenv("AZURE_OPENAI_API_KEY")
model          = os.getenv("MODEL_NAME")
search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT", "")  # env 표준 변수명
search_key     = os.getenv("AZURE_SEARCH_KEY")            # env 표준 변수명
index_name     = "telecom-terms-index"
pdf_path       = "../../data/pdf/InternetServiceToU.pdf"

if ".openai.azure.com" in endpoint and "/openai/deployments/" not in endpoint:
    endpoint = f"{endpoint}/openai/deployments/{model}"

print(f"Inference Endpoint : {endpoint}")
print(f"Model              : {model}")
print(f"Search Endpoint    : {search_endpoint}")

Inference Endpoint : https://kt-new-foundry-resource.openai.azure.com/openai/deployments/gpt-5-nano
Model              : gpt-5-nano
Search Endpoint    : https://kt-ai-search-260226.search.windows.net


In [ ]:
# 강사용: 임베딩 생성 코드. 실행하지 마세요!
# 이미 생성된 벡터저장소를 조회만 하시면 되세요!
# import sys, subprocess
# subprocess.run([sys.executable, "01_rag_ingest.py"])

임베딩 및 문서 준비: 100%|██████████| 450/450 [01:29<00:00,  5.04it/s]


450개 데이터 업로드 중...
하이브리드 인덱싱 완료!


CompletedProcess(args=['/anaconda/envs/kt_env/bin/python', '01_rag_ingest.py'], returncode=0)

## 2. 데이터 수집 (Ingestion)


In [4]:
def extract_text_with_pages(file_path):
    print(f"Extracting text from {file_path}...")
    loader = PyMuPDFLoader(file_path)
    docs = loader.load()
    pages = []
    for i, doc in enumerate(docs):
        if doc.page_content:
            pages.append({"page_num": i + 1, "content": doc.page_content})
    return pages

if os.path.exists(pdf_path):
    pages_data = extract_text_with_pages(pdf_path)
    print(f"Extracted {len(pages_data)} pages.")
else:
    print(f"Error: PDF file not found at {pdf_path}. Please check the path.")

Extracting text from ../../data/pdf/InternetServiceToU.pdf...
Extracted 232 pages.


## 3. 인덱스 생성 및 데이터 업로드

**Azure AI Search**에 데이터를 저장하려면 '인덱스(Index)'를 정의해야 합니다. 인덱스는 데이터베이스의 테이블과 비슷합니다.

**필드 구성:**
- `id`: 각 데이터 조각의 고유 식별자
- `content`: 실제 텍스트 내용 (검색 대상, 한국어 분석기 `ko.microsoft` 적용)
- `source`: 파일명 등 출처 정보
- `page`: 페이지 번호 (답변 출처 표시에 사용)

In [ ]:
def create_and_populate_index(pages_data):
    """
    Azure AI Search 인덱스를 생성하고 PDF 데이터를 업로드합니다.
    최신 azure-search-documents SDK 사용법에 맞춰 작성되었습니다.
    """
    if not search_endpoint:
        raise ValueError("AZURE_SEARCH_ENDPOINT 환경 변수가 설정되지 않았습니다.")
    
    # 1. 클라이언트 초기화
    credential = AzureKeyCredential(search_key) if search_key else DefaultAzureCredential()
    index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)
    
    # 2. 인덱스 스키마 정의
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String, analyzer_name="ko.microsoft"),
        SimpleField(name="source", type=SearchFieldDataType.String),
        SimpleField(name="page", type=SearchFieldDataType.Int32),
    ]
    
    index = SearchIndex(name=index_name, fields=fields)
    print(f"Creating/Updating index '{index_name}'...")

    try:
        index_client.create_index(index)
        print(f"인덱스 '{index_name}' 생성 완료")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"인덱스 '{index_name}' 이미 존재 → 기존 인덱스 사용")
        else:
            raise e
    
    # 3. 문서 청킹 및 준비
    documents = []
    doc_id = 0
    
    for page in pages_data:
        page_num = page["page_num"]
        content = page["content"]
        
        # 간단한 청킹 
        chunk_size = 800
        chunks = [content[i:i+chunk_size] for i in range(0, len(content), chunk_size)]
        
        for chunk in chunks:
            documents.append({
                "id": f"doc_{doc_id}",
                "content": chunk,
                "source": "InternetServiceToU.pdf",
                "page": page_num
            })
            doc_id += 1
    
    # 4. 데이터 업로드
    search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=credential)
    print(f"Uploading {len(documents)} chunks...")
    search_client.upload_documents(documents)
    
    print("Waiting for indexing...")
    time.sleep(2)
    print("Ingestion complete!")

# 인덱싱 실행
if 'pages_data' in locals():
    create_and_populate_index(pages_data)

Creating/Updating index 'telecom-terms-index-notebook'...
인덱스 'telecom-terms-index-notebook' 이미 존재 → 기존 인덱스 사용
Uploading 341 chunks...
Waiting for indexing...
Ingestion complete!


## 4. 검색 (Retrieval)

이제 데이터가 Azure AI Search에 저장되었습니다. 사용자의 질문과 관련된 문서를 검색해 봅니다.
여기서는 간단한 **키워드 검색**을 사용하지만, Azure AI Search는 벡터 검색(Vector Search)과 하이브리드 검색도 지원합니다.

In [14]:
def retrieve_documents(query):
    """
    Azure AI Search를 사용하여 사용자 쿼리와 관련된 문서를 검색합니다.
    최신 azure-search-documents SDK 사용법에 맞춰 작성되었습니다.
    """
    if not search_endpoint:
        raise ValueError("AZURE_SEARCH_ENDPOINT 환경 변수가 설정되지 않았습니다.")
    
    credential = AzureKeyCredential(search_key) if search_key else DefaultAzureCredential()
    search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=credential)
    
    print(f"Searching for: '{query}'")
    
    # select를 통해 필요한 필드만 가져옵니다.
    results = search_client.search(search_text=query, top=3, select=["content", "page"])
    
    context = ""
    sources = set()
    for result in results:
        page_num = result.get("page")
        context += f"---\n[Page {page_num}] {result['content']}\n"
        if page_num:
            sources.add(str(page_num))
        
    return context, sorted(list(sources))

# 검색 테스트
user_query = "giga wifi가 뭐야?"
context, sources = retrieve_documents(user_query)
print(f"\n[Retrieved Context Sample]:\n{context[:200]}...") # 앞부분만 출력
print(f"[Sources]: {sources}")

Searching for: 'giga wifi가 뭐야?'

[Retrieved Context Sample]:
---
[Page 84] -  84 - 
3년 약정 
월 1,100원 
※ WiFi 패키지 plus는 해지 시까지 무상 A/S가 제공되며, 또한 임대료는 해지 시까지 부과되며(3년 이후 무상임대 
아님) 해지 시 AP는 반납해야 합니다.  
※ 해당 상품의 약정은 인터넷 약정 기준으로 제공됩니다. 
GiGA WiFi Buddy 
무약정 
월 8,800원 
...
[Sources]: ['189', '84']
-  1 - 
 
 
 
KT 인터넷 이용약관 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
2025. 9


In [18]:
print("\n".join(p["content"] for p in pages_data))

-  1 - 
 
 
 
KT 인터넷 이용약관 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
2025. 9
-  2 - 
제1장  총   칙 
 
제 1조 ( 약관의 적용 ) 
① 주식회사 케이티(이하 "케이티"라 합니다)의 인터넷서비스(이하 "서비스"라 합니다)를 
이용할 때는 이 약관과 전기통신서비스이용기본약관(이하 “기본약관”이라 합니다)을 
함께 적용합니다. 
② 케이티인터넷서비스 이용약관은 지정된 홈페이지(www.kt.com)에 게시하여 공지합니다. 
 
제 2조 ( 정의 ) 
기본약관에서 정의한 용어 이외에 이 약관에서 사용하는 용어의 정의는 다음과 같습니다. 
1. 
KT internet: 케이티가 제공하는 초고속인터넷서비스 
2. 
KT biz kornet: 케이티가 구축한 인터넷망의 이름 또는 인터넷 전용회선서비스 
3. 
KT WiFi: 노트북, PDA, TABLET 등의 단말을 이용하여 가정, 기업, KT WiFi ZONE 
등에서 무선으로 초고속인터넷을 이용할 수 있는 서비스 
4. 
지니 TV VOD: KT internet 회선종단에 지니 TV 단말을 설치하고, 가정내 정보단말 
및 각종기기를 유무선으로 연결하여 고품질 VOD 및 홈네트워킹 등 다양한 
서비스를 이용할 수 있는 서비스 
5. 
고객 ID: 고객을 식별하고 서비스를 이용하게 하기 위하여 고객이 선정하고 
케이티가 부여하는 문자와 숫자 등의 조합  
6. 
비밀번호: 고객의 서비스 이용권을 보호하기 위하여 고객이 선정하는 문자와 숫자 
등의 조합 
7. 
IP주소(Internet Protocol address): 인터넷 망에서 송신원과 송신선을 식별하기 위해 
배정하는 주소 
8. 
IP단말기기: IP번호를 지니는 단말기기 
9. 
유동IP: 특정 고객에 대해 IP주소가 고정적으로 배정되는 것이 아니라 고객이 KT 
biz kornet에 접속할 때마다 새로이 할당되는 IP주소 
10. xDSL 방식: 일반 전화망의 주파수 대역 중 비 사용중인 상위 대역을 이용하

## 5. 답변 생성 (Generation)

마지막으로 검색된 문맥(Context)과 사용자의 질문을 LLM(Large Language Model)에게 전달하여 최종 답변을 생성합니다.
이때 시스템 프롬프트를 통해 AI의 페르소나(상담원)를 설정합니다.

In [13]:
def generate_rag_response(query, context):
    """검색된 컨텍스트를 바탕으로 LLM 답변을 생성합니다."""
    if not endpoint:
        raise ValueError("END_POINT 환경 변수가 설정되지 않았습니다.")

    credential = AzureKeyCredential(api_key) if api_key else DefaultAzureCredential()
    client = ChatCompletionsClient(endpoint=endpoint, credential=credential)

    system_prompt = """당신은 통신사 고객 상담을 담당하는 친절하고 전문적인 AI 상담원입니다.
    제공된 [약관 내용]을 바탕으로 고객의 질문에 정확하게 답변해 주세요.
    약관에 없는 내용은 \"죄송하지만 해당 내용은 약관에서 찾을 수 없습니다.\"라고 답변하세요.
    답변은 한국어로 정중하게 작성해 주세요."""

    user_message = f"[약관 내용]\n{context}\n\n[고객 질문]\n{query}"

    response = client.complete(
        messages=[
            SystemMessage(content=system_prompt),
            UserMessage(content=user_message),
        ]
    )
    return response.choices[0].message.content

# 최종 실행
if context:
    print("답변 생성 중...")
    answer = generate_rag_response(user_query, context)
    print(f"\n상담원: {answer}")
    if sources:
        print(f"\n(출처: 약관 {', '.join(sources)} 페이지)")
else:
    print("관련된 정보를 찾을 수 없습니다.")

답변 생성 중...

상담원: GiGA WiFi란 KT의 인터넷 서비스와 연계되어 제공되는 무선 인터넷 공유 장치(AP)를 빌려 사용하는 WiFi 서비스예요. 간단히 말해, 인터넷 연결은 KT 인터넷으로 이용하고, 그 연결을 와이파이로 쓸 수 있게 해 주는 휴대용 공유기 같은 장비를 임대하는 서비스입니다.

약관에 따른 주요 내용
- 주요 상품: GiGA WiFi Buddy, GiGA WiFi Buddy ax
- 약정 형태: 무약정, 1년/2년/3년 약정 옵션이 있습니다.
- 요금 및 임대: 약정에 따라 월 임대료가 달라지며, 임대료는 해지 시까지 부과됩니다. 해지 시에는 AP를 반납해야 합니다.
- A/S: WiFi 패키지 plus와 GiGA WiFi Buddy(또는 Buddy ax) 등의 경우 해지 시까지 무상 A/S가 제공됩니다.
- KT 인터넷과 함께 GiGA WiFi Buddy를 사용하는 경우에는 이를 “인터넷 와이드”라고 부르기도 합니다.
- 예시 요금(약정별, 개별 상품마다 차이가 있음)
  - GiGA WiFi Buddy
    - 무약정: 월 8,800원
    - 1년 약정: 월 7,150원
    - 2년 약정: 월 5,500원
    - 3년 약정: 월 1,100원
  - GiGA WiFi Buddy ax
    - 무약정: 월 8,800원
    - 1년 약정: 월 7,150원
    - 2년 약정: 월 5,500원
    - 3년 약정: 월 1,650원
- 참고: 위 내용은 약관에 기재된 상품별 요금 및 약정 형태의 예시이며, 상세한 금액은 해당 상품의 약정 유형에 따라 다릅니다.

추가로 확인하고 싶은 특정 상품이 있으면 말씀해 주세요. 약관 기준으로 비교해 드리겠습니다.

(출처: 약관 189, 84 페이지)
